In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
grades_per_exam = pd.read_csv('./Grades/grades_per_exam.csv',index_col=0)
grades_per_exam

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
from xgboost import XGBRegressor

In [ ]:
quedan_e = pd.read_csv('./Abandono/mastbias_e{}.csv'.format(week_exam[w]),index_col=0)
quedan_e

In [ ]:
def nota_final_pipeline(w, indicadores):
  theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)
  w_ind2 = ind.loc[theta.index]
  w_ind2['theta'] = theta['conf1']

  week_exam = {1:1, 2:1, 3:2, 4:3, 5:4, 6:5, 7:5}
  print('Exam:',week_exam[w])
  quedan_e = pd.read_csv('./Abandono/mastbias_e{}.csv'.format(week_exam[w]),index_col=0)

  w_ind = w_ind2.loc[w_ind2.index]


  w_ind['grade'] = grades_per_exam['Course grade']
  w_ind['grade'] = w_ind['grade'].fillna(0)

  if 'CFA' in w_ind.columns:
      w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  if 'avg_grade' in w_ind.columns:
      w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  if 'avg_attempts' in w_ind.columns:
      w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)


  X = w_ind.drop(['grade'],axis=1)
  y = w_ind['grade']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
  print('len(X), len(y_test)',len(X), len(y_test))

  X_noatt = X.loc[quedan_e.index]
  y_noatt = w_ind.loc[quedan_e.index,'grade']
  print('Quedan:',len(X_noatt))
  X_train_no, X_test_no, y_train_no, y_test_no = train_test_split(X_noatt, y_noatt, test_size=0.25, random_state=42)
  print('len(X_no), len(y_test_no)',len(X_noatt), len(y_test_no))

  pipe_xgb = Pipeline([('scl', StandardScaler()),
                      ('XGB', XGBRegressor())])

  xgb_param_grid = [{'XGB__learning_rate': [0.1],
                      'XGB__max_depth': [5],
                      'XGB__min_child_weight': [2.0],
                      'XGB__subsample': [1.0],
                      'XGB__n_estimators': [75]}]

  xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)
  xgb_grid_search2 = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)

  grids = [xgb_grid_search, xgb_grid_search2]
  names = ['xgb_att','xgb_noatt']
  xgb_grid_search.fit(X_train,y_train)
  xgb_grid_search2.fit(X_train_no,y_train_no)

  mses = {}
  c = 0
  for i in grids:
    mses[names[c]] = -(i.score(X_test_no,y_test_no))
    print(names[c], len(y_test_no))
    c+= 1

  return mses

In [ ]:
rmses = {}
for w in range(1,8):
  print('\nWeek:', w)
  ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
  rmse =  nota_final_pipeline(w, indicadores = ind)
  rmses['Week {}'.format(str(w))] = rmse
  print(rmses['Week {}'.format(str(w))])

In [ ]:
rmses

In [ ]:
ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)

theta = pd.read_csv('./Modelos/Cummulative/Configuración al menos un evaluable/theta_one_week{}.csv'.format(w),index_col=0)
w_ind2 = ind.loc[theta.index]
w_ind2['theta'] = theta['conf1']

week_exam = {1:1, 2:1, 3:2, 4:3, 5:4, 6:5, 7:5}
print('Exam:',week_exam[w])
quedan_e = pd.read_csv('./Abandono/mastbias_e{}.csv'.format(week_exam[w]),index_col=0)

w_ind = w_ind2.loc[w_ind2.index]
print('Quedan:',len(w_ind))

w_ind['grade'] = grades_per_exam['Course grade']
w_ind['grade'] = w_ind['grade'].fillna(0)

if 'CFA' in w_ind.columns:
    w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
if 'avg_grade' in w_ind.columns:
    w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
if 'avg_attempts' in w_ind.columns:
    w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)


X = w_ind.drop(['grade'],axis=1)
y = w_ind['grade']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


pipe_xgb = Pipeline([('scl', StandardScaler()),
                    ('XGB', XGBRegressor())])

xgb_param_grid = [{'XGB__learning_rate': [0.1],
                    'XGB__max_depth': [5],
                    'XGB__min_child_weight': [2.0],
                    'XGB__subsample': [1.0],
                    'XGB__n_estimators': [75]}]

xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
        param_grid=xgb_param_grid,
        scoring='neg_mean_squared_error',
        cv=3)
grids = [xgb_grid_search]
names = ['xgb']
n = 0
for pipe in grids:
  print(names[n])
  pipe.fit(X_train,y_train)
  n+=1


w_ind2['grade'] = grades_per_exam['Course grade']
w_ind2['grade'] = w_ind2['grade'].fillna(0)

if 'CFA' in w_ind2.columns:
    w_ind2['CFA'] = w_ind2['CFA'].replace('0 problems attempted',0)
if 'avg_grade' in w_ind2.columns:
    w_ind2['avg_grade'] = w_ind2['avg_grade'].replace('0 problems attempted',0)
if 'avg_attempts' in w_ind.columns:
    w_ind2['avg_attempts'] = w_ind2['avg_attempts'].replace('0 problems attempted',0)

quedan_e = pd.read_csv('./Abandono/mastbias_e{}.csv'.format(week_exam[7]),index_col=0)
w_ind3 = w_ind2.loc[quedan_e.index]

X_test2 = w_ind3.drop(['grade'],axis=1)
y_test2 = w_ind3['grade']
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_test2, y_test2, test_size=0.25, random_state=42)
y_test3 = pd.DataFrame(columns = ['grade'])
for i in y_test2.index:
  if i not in y_train.index:
    y_test3.loc[i,'grade'] = y_test2.loc[i,'grade']
print(len(y_test2))
print(len(y_train))
#y_test2.drop[i if i in y_test2 else for i in y_train.index]
print('X debería tener los que quedan:,',len(X), len(quedan_e))
print('y_test debería tener 8708:,',len(y_test))


mses = {}
grid_dict = {0: 'XGBoost'}
for i, model in enumerate(grids):
  mses = -(model.score(X_test2,y_test2))

In [ ]:
y_test3.loc[i] = 0

In [ ]:
y_test3 = pd.DataFrame(columns = ['grade'])
for i in y_test2.index:
  print(i)
  if i not in y_train.index:
    y_test3.loc[i] = y_test2.loc[i]

In [ ]:
y_test3

In [ ]:
y_test2.index

In [ ]:
y_train

In [ ]:
quedan_e